# 06 · Energy resolution at Q_bb (0νββ)

**Purpose.** Energy of 0νββ events with the fibers and with the PMTs, and the resolution at Q_bb = 2458 keV.

**Inputs.** `/data/marian/IC_ANALYSIS/hypathia/<fibers|pmts>/g<gain>/` (42 files × 300 nexus events, 10 bar, `SECTOR_AREA`), and the Kr kDSTs of the same gain in `dorothea/<det>/g<gain>/` for the (x, y) map and the energy scale.

**Outputs.** Energy per event (cached in `work/qbb/`), resolution at Q_bb, comparison with the E^-1/2 extrapolation of the Kr result (05).

**What is checked.** truth containment (`/MC/hits`), the light response maps (light table vs Kr), lifetime and (x, y) corrections per S2 slice, fiducial cuts, Gaussian fit of the peak.

**Why not dorothea.** Its Kr cuts reject most 0νββ events (S1 height ≤ 30 pes, S2 width ≤ 100 µs) and it treats each event as a point. Here every S2 slice is corrected on its own.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

### Choose the run
Change only the next cell. The first run reads all PMaps (about 10 s per file and detector) and caches one table per file in `work/qbb/`; later runs take seconds. Changing `TAU_MS` or `GAIN` makes new cache files.

In [ ]:
# ---- Choose what to look at (change only this cell) ----
SAMPLE = '20250205_Next100_bb_300ev_ELoff'     # 0nubb, 42 files x 300 events
GAIN   = 1050          # EL gain of the run (only 1050 exists for this sample)
TAU_MS = 12.0          # lifetime used in the correction: 12 ms = detsim input; 05 fits 12.23 (fibers), 12.84 (PMTs) on Kr

# Fiducial cuts on the reconstructed track (slices with >= 5 pes of SiPM charge)
R_MAX = 400            # mm, largest slice radius (active radius 490 mm)
Z_MIN, Z_MAX = 20, 1167   # mm, drift length 0-1187 mm (Kr kDST)
E_COL = 'e_kr'         # energy used: 'e_kr' (Kr map, as NEXT does with data) or 'e_table' (light table); see below
FILES = None           # None = all files; e.g. [2, 3] to try a few first

In [ ]:
paths.show(SAMPLE)    # what exists for this sample

### Method
For every event and every S2 slice (~1 µs) of the hypathia PMaps:
1. **t0** = time of the largest S1 slice (the decay is at 100 µs in detsim). Drift time dt = t_slice − t0, z = dt × 1 mm/µs.
2. **Lifetime**: E × exp(dt/τ).
3. **(x, y)**: charge barycentre of the SiPMs in that slice. Correction E / K(x, y), with K from
   - `e_table`: the S2 light table detsim used (sum over sensors, IC's own table classes), normalised to the centre;
   - `e_kr`: the Kr kDSTs of the same detector and gain (median of the lifetime-corrected S2e in 20 mm bins, normalised to R < 30 mm), as NEXT does with data.
4. Event energy = sum over slices and S2 peaks. Energy scale: 41.5 keV ↔ the Kr centre value (`qbb.kr_centre`).

Tried and left out: sharing each slice among its SiPMs ("hits") and correcting each share at the SiPM position. On file 2 it over-corrects the fibers (spread 0.36 % vs 0.23 % at R < 400 mm) and helps the PMTs only near the wall.

Code: `icana/qbb.py`.

In [ ]:
from icana import qbb, plots
plots.set_style()
SAVE = False         # True: also save each figure to work/figures/ (PDF for the thesis, PNG)
def show(fig, name):
    if SAVE: print("saved", plots.save(fig, name))
    plt.show()

TAU_US = TAU_MS * 1e3
EV = {}
for det in ["fibers", "pmts"]:
    EV[det] = qbb.all_events(det, GAIN, SAMPLE, TAU_US, files=FILES)
    EV[det]["e_kev"] = EV[det][E_COL] * 41.5 / qbb.kr_centre(det, GAIN, TAU_US)
    print(f"{det}: {len(EV[det])} events with PMaps from {EV[det].file.nunique()} files")

### Light response maps
Response to a point-like S2 at (x, y), relative to the centre: light table (lines) and Kr map (points), along the x axis. The fibers are almost flat (within a few % up to R ≈ 440 mm); the PMTs lose ~9 % at R = 400 mm. The Kr maps are also shown in 2D.

In [ ]:
r = np.linspace(0, 485, 98)
fig, ax = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
for det, col in [("fibers", "r"), ("pmts", "k")]:
    lt, kr = qbb.lce_map(det), qbb.kr_map(det, GAIN, TAU_US)
    ax[0].plot(r, lt(r, 1), "-", color=col, label=f"{det}, light table")
    ax[0].plot(r, kr(r, 1), "o", color=col, ms=4, label=f"{det}, Kr map")
ax[0].set_xlabel("x (mm), y = 0"); ax[0].set_ylabel("response / centre"); ax[0].set_ylim(0.6, 1.1); ax[0].grid(); ax[0].legend(fontsize=14)
g = np.linspace(-490, 490, 197); X, Y = np.meshgrid(g, g, indexing="ij")
for a, det in zip(ax[1:], ["fibers", "pmts"]):
    m = qbb.kr_map(det, GAIN, TAU_US)(X, Y).reshape(X.shape)
    im = a.pcolormesh(X, Y, m, vmin=0.8, vmax=1.05, shading="auto"); a.set_aspect("equal"); a.set_title(f"Kr map, {det}")
    a.set_xlabel("x (mm)"); a.set_ylabel("y (mm)")
fig.colorbar(im, ax=ax[1:], label="response / centre")
show(fig, f"qbb_g{GAIN}_response_maps")

### Truth: how much of the 2458 keV stays in the active volume
`e_true` = energy of the MC hits in ACTIVE. Events below Q_bb lost a bremsstrahlung photon or X-ray out of the active volume (or the track left it).

In [ ]:
e = EV["fibers"]     # same nexus events for both detectors
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(e.e_true, np.linspace(0, 2500, 101), histtype="step", lw=2)
ax.set_yscale("log"); ax.set_xlabel("true energy in ACTIVE (keV)"); ax.set_ylabel("events"); ax.grid()
show(fig, f"qbb_true_energy")
full = e.e_true > 2450
print(f"{full.mean()*100:.1f} % of {len(e)} events keep > 2450 keV in ACTIVE")

### Selection
Fiducial cuts from the top cell, on the reconstructed slices. The purity column is the fraction of selected events that really kept > 2450 keV (truth).

In [ ]:
def select(ev):
    cuts = {"with S1 and S2": (ev.n_s1 > 0) & (ev.n_s2 > 0),
            f"r_max < {R_MAX} mm": ev.r_max < R_MAX,
            f"z in [{Z_MIN}, {Z_MAX}] mm": (ev.z_min > Z_MIN) & (ev.z_max < Z_MAX)}
    keep = np.ones(len(ev), bool); rows = []
    for name, c in cuts.items():
        keep &= c.values
        rows.append(dict(cut=name, events=int(keep.sum()), efficiency=keep.mean(), purity=(ev.e_true[keep] > 2450).mean()))
    return ev[keep], pd.DataFrame(rows)
SEL = {}
for det in ["fibers", "pmts"]:
    SEL[det], table = select(EV[det]); print(det); print(table.round(3).to_string(index=False))

### Energy spectrum and resolution at Q_bb
Gaussian fit in a window around the peak (−1.5 % to +1 % of the median of the events above 2350 keV; the low side has a tail from energy that escaped or was not corrected). For each detector: raw, lifetime-corrected and fully corrected energy, and the same fit on truth-contained events only (`e_true > 2450 keV`) as a reference.

In [ ]:
def fit_qbb(e_kev, lo=0.985, hi=1.01, nbins=60):
    m = np.median(e_kev[e_kev > 2350])
    return fits.fit_peak(e_kev, np.linspace(lo * m, hi * m, nbins))

RES = {}
for det in ["fibers", "pmts"]:
    s = SEL[det]; scale = 41.5 / qbb.kr_centre(det, GAIN, TAU_US)
    rows = {}
    for col, name in [("e_raw", "raw"), ("e_lt", "lifetime"), ("e_table", "lifetime + table (x, y)"), ("e_kr", "lifetime + Kr (x, y)")]:
        f = fit_qbb(s[col].values * scale); rows[name] = [f["mu"], f["fwhm_pct"], f["fwhm_pct_err"]]
    f = fit_qbb(s.e_kev[s.e_true > 2450].values); rows[f"{E_COL}, truth-contained only"] = [f["mu"], f["fwhm_pct"], f["fwhm_pct_err"]]
    RES[det] = fit_qbb(s.e_kev.values)
    print(det); print(pd.DataFrame(rows, index=["peak (keV)", "FWHM (%)", "± (fit)"]).T.round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 7))
for det, col in [("fibers", "r"), ("pmts", "k")]:
    ax.hist(SEL[det].e_kev, np.linspace(2200, 2550, 141), histtype="step", lw=2, color=col, label=f"{det} ({len(SEL[det])} events)")
ax.set_yscale("log"); ax.set_xlabel("energy (keV)"); ax.set_ylabel("events"); ax.legend(); ax.grid()
show(fig, f"qbb_g{GAIN}_spectrum")
for det in ["fibers", "pmts"]:
    show(plots.peak_fit(RES[det], f"{det}, R < {R_MAX} mm", "0νββ events", xlabel="energy (keV)"), f"qbb_g{GAIN}_{det}_peak")

### Residual radial dependence
Median energy of truth-contained events in bins of the true largest radius of the track, relative to the innermost bin. A flat line means the (x, y) correction works. Seen on file 2: flat within 0.3 % for fibers, a drop of up to ~2 % near the wall for PMTs.

In [ ]:
bins = [0, 150, 250, 300, 350, 400, 450]
rows = {}
for det in ["fibers", "pmts"]:
    c = EV[det][(EV[det].e_true > 2450) & (EV[det].r_max < 480)]
    g = c.groupby(pd.cut(c.r_true_max, bins))[["e_raw", "e_kr", "e_table"]].median()
    rows[det] = (g / g.iloc[0]).round(4)
pd.concat(rows, axis=1)

### Comparison with the Kr result
If the resolution were pure photon statistics + Fano, it would scale as E^-1/2: R(Q_bb) = R(Kr) × √(41.5 / 2458). Kr numbers from `05_dorothea_kr` (gain 1050, R < 30 mm, lifetime corrected, 2026-11-22).

In [ ]:
KR_FWHM = {"fibers": 3.07, "pmts": 3.46}     # % FWHM at 41.5 keV, from 05 (gain 1050)
pd.DataFrame({det: {"Kr FWHM (%)": KR_FWHM[det],
                    "E^-1/2 extrapolation to Q_bb (%)": KR_FWHM[det] * np.sqrt(41.5 / 2458),
                    "measured at Q_bb (%)": RES[det]["fwhm_pct"], "± (fit)": RES[det]["fwhm_pct_err"],
                    "peak (keV)": RES[det]["mu"], "events in fit window": int(RES[det]["y"].sum())}
              for det in ["fibers", "pmts"]}).T.round(3)

### Notes
- The fiber S1 is a placeholder in detsim (see 03). Here S1 only gives t0, so this does not matter.
- The PMT residual radial dependence (above) widens the PMT peak; `R_MAX` controls how much of it enters.
- The peak position in keV tests the linearity between 41.5 keV and Q_bb; it is not forced to 2458.
- Statistics: ~12 600 nexus events; the fiducial selection keeps a fraction of them (table above). The ± is the fit error only.